# Notebook 08 — Stage 4: analysis audit + PASS/REVIEW/FAIL verdict

**Final stage. Analyst-facing decision layer.** Reads Stage 3's
`enhanced.csv` and produces the analysis-ready CSV plus a per-row
verdict (`analysis_audit_status` = PASS / REVIEW / FAIL) with explicit
reason codes.

```text
07_stage3.ipynb
   └── <stage3_out>/data/enhanced.csv
                            │
                            ▼
                 08_stage4.ipynb            ◄── THIS NOTEBOOK
                    • re-resolve canonical aliases (defensive)
                    • coerce + standardise types
                    • missing-key + duplicate detection
                    • range checks (long-format output)
                    • strict DIC species audit (gate, not diagnostic)
                    • PASS / REVIEW / FAIL classification with reason codes
                    • write analysis_ready.csv + all audit tables
```

**Severity ladder** (the analyst's decision logic):

- **FAIL** — missing required key, Stage 3 strict carbonate issue,
  strict DIC species-sum fail, DIC unit mismatch, unknown solver,
  unknown input pair.
- **REVIEW** — duplicate complete-key row, range flag, Stage 3
  (non-strict) carbonate issue, Stage 2 replicate conflict, DIC unit
  *missing* (vs mismatched), Stage 3 robust DIC outlier, Stage 3 pH
  diagnostic mismatch.
- **PASS** — none of the above.

This is the PASS / WARN / FAIL "quality gate" pattern (NDepend,
SonarQube, UN/ABS *Data Quality Manual Part B*). We use REVIEW for the
middle tier because that's what an analyst actually does: review the
row, decide whether to include it. **Flags are advisory** — nothing is
deleted; the analyst filters on `analysis_audit_status` downstream.

**Stage 4 does NOT** rebuild chemistry, re-filter sample rows, or
overwrite Stage 3's QC flags. It audits, classifies, and writes.


## Parameters

Single tagged `parameters` cell. The default `INPUT_CSV` points at the
new short Stage 3 path — the original's
`oa_stage3_outputs\stage3_enhanced.csv` was the **sixth and final**
audit-flagged path bug (the refactored Stage 3 writes `<stage3_out>/data/enhanced.csv`).


In [ ]:
# =====================================================================
# Parameters cell  (papermill tag: "parameters")
# =====================================================================

# --- I/O -------------------------------------------------------------
INPUT_CSV = r"C:\Users\OA_2023-03\OneDrive\Habitat Suitabilty model\OA\data_1\oa_stage3_outputs\data\enhanced.csv"
OUT_DIR = r"C:\Users\OA_2023-03\OneDrive\Habitat Suitabilty model\OA\data_1\oa_stage4_outputs"

# --- Config override (optional) ------------------------------------
# Deep-merges onto oa_stage4.STAGE4_DEFAULTS.
CONFIG_PATH = None

# --- Stage 4 behaviour ---------------------------------------------
NO_RANGE_CHECKS = False       # Skip range checks entirely.
NO_DIC_SPECIES_CHECK = False  # Skip the strict DIC audit.
NO_PARQUET = False            # Skip Parquet writes (CSV only).
DRY_RUN = False               # Plan everything, write nothing.


## Setup

All Stage 4 logic — coerce-and-standardise, missing-key/duplicate
detection, range checks, the strict DIC audit, and the PASS/REVIEW/FAIL
classifier — lives in `oa_stage4.py`. The notebook is thin
orchestration.

The `RangePolicy` dataclass is **imported once from `oa_policy.py`** —
no per-notebook redefinition. The audit identified this as the
third-stage redefinition with *different fields*; the unified dataclass
in `oa_policy.py` includes every field every stage needs.


In [ ]:
from __future__ import annotations

import sys
from dataclasses import asdict
from pathlib import Path

import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = None

from oa_common import (
    deep_update,
    die,
    ensure_dir,
    md_table_from_df,
    normalize_columns,
    utc_stamp,
    write_csv_and_parquet,
    write_json,
    write_text,
)
from oa_policy import policy_from_config
from oa_schema import load_config
from oa_stage2 import (
    ensure_required_columns,
    ensure_stage2_dirs,
    make_column_inventory,
    make_presence_table,
    materialize_canonical_aliases,
)
from oa_stage4 import (
    STAGE4_DEFAULTS,
    DicSpeciesAudit,
    add_readiness_status,
    coerce_and_standardize,
    detect_duplicates,
    dic_species_audit,
    missing_key_rows,
    reason_count_table,
    run_range_checks,
)


## Load input and canonicalise

Defensive: re-resolve aliases (no-op when fed Stage 3's output, recovers
when fed something else), check required columns, coerce types,
re-normalise scale/unit strings.


In [ ]:
input_csv = Path(INPUT_CSV).expanduser().resolve()
if not input_csv.exists():
    die(f"File not found: {input_csv}\n"
        f"Did Stage 3 run successfully? Stage 4 reads its 'enhanced.csv'.")
if input_csv.suffix.lower() not in {".csv", ".txt"}:
    die(f"Expected a CSV-like file, got: {input_csv.name}")

# Merge: STAGE4_DEFAULTS <- user CONFIG_PATH override.
user_config, config_source = load_config(CONFIG_PATH)
if CONFIG_PATH:
    config = deep_update(STAGE4_DEFAULTS, {
        k: v for k, v in user_config.items()
        if k in STAGE4_DEFAULTS or k not in user_config
    })
else:
    config = STAGE4_DEFAULTS

out_root = ensure_dir(Path(OUT_DIR).expanduser().resolve())
dirs = ensure_stage2_dirs(out_root)   # same {data,tables,reports,logs} layout

notes: list[str] = []

df = pd.read_csv(input_csv)
df = normalize_columns(df)
df["source_file_stage4"] = str(input_csv)
df["stage4_processed_utc"] = utc_stamp()

df, alias_resolution = materialize_canonical_aliases(df, config["canonical_aliases"], notes)
ensure_required_columns(df, config["required_stage3_columns"])
df = coerce_and_standardize(df, notes)

presence_df = make_presence_table(
    df,
    required=config["required_stage3_columns"],
    expected=config["expected_stage3_columns"],
)
missing_opt = presence_df.loc[
    (~presence_df["required"]) & (~presence_df["present"]), "column"
].tolist()
if missing_opt:
    notes.append("Optional expected columns not found: " + ", ".join(missing_opt))

inventory_df = make_column_inventory(df)

print(f"Rows loaded: {len(df):,}")
print(f"Columns    : {df.shape[1]}")

if display is not None:
    display(presence_df)


## Key integrity and duplicate detection

Two related but distinct checks:

- **`missing_key_rows`** — rows where any duplicate-key column is
  null or empty. These get one `flag_missing_key__<col>` per missing
  column in the audit table.
- **`detect_duplicates`** — Stage 4's stricter duplicate detector: a
  row is duplicate only if **every** key column is non-null AND the
  values collide with another row's. Rows with any null in the key
  fall through to the missing-key check, not the duplicate check.
  (This is stricter than Stage 2's flag, which is advisory and works
  on partial-key collisions.)


In [ ]:
dup_keys = [str(k).strip() for k in config.get("duplicate_keys", []) if str(k).strip()]
key_missing = missing_key_rows(df, dup_keys)
dup_table, dup_msgs, keys_used = detect_duplicates(df, dup_keys)

print(f"Missing-key rows: {len(key_missing):,}")
print(f"Duplicate rows  : {len(dup_table):,}")
for m in dup_msgs:
    print(f"  {m}")

if display is not None:
    if not key_missing.empty:
        display(key_missing.head(20))
    if not dup_table.empty:
        display(dup_table.head(20))


## Range checks (long-format output)

Unlike Stages 1A/1B (which write `flag_*_out_of_range` *columns* on
the row-level frame), Stage 4 produces:

- `range_check_summary` — one row per logical variable
  (`salinity`, `temperature`, `observed_ph`, …) with `min_allowed`,
  `max_allowed`, `n_valid`, `n_flagged`.
- `range_flags_long` — one row per (row_index, variable, direction)
  violation, with the offending value and the ID columns attached.

Long format is the right shape for analyst tooling: "show me every
salinity below 30 in 2024" is one filter; the row-level boolean
columns would need unpivoting first.

`RangePolicy` is the unified dataclass from `oa_policy.py`. Stage 4's
config supplies *wider* bounds than Stage 1A/1B (this stage asks "is
this physically possible seawater chemistry?" not "is this typical
open-ocean chemistry?").


In [ ]:
range_summary = pd.DataFrame()
range_flags_df = pd.DataFrame()
policy = policy_from_config(config)

if not NO_RANGE_CHECKS:
    range_summary, range_flags_df = run_range_checks(df, policy)

    # Attach per-row range_flag_count for the readiness classifier.
    if not range_flags_df.empty:
        cnt = range_flags_df.groupby("row_index").size().rename("range_flag_count")
        df = df.merge(cnt, left_index=True, right_index=True, how="left")

    print(f"Range variables checked: {len(range_summary):,}")
    print(f"Flagged cells          : {len(range_flags_df):,}")
else:
    notes.append("Range checks were disabled.")

df["range_flag_count"] = (
    pd.to_numeric(df["range_flag_count"], errors="coerce")
    .fillna(0).astype(int)
    if "range_flag_count" in df.columns
    else pd.Series(0, index=df.index, dtype=int)
)

if display is not None and not range_summary.empty:
    display(range_summary)


## Strict DIC species audit

Stricter than Stage 3's diagnostic version because:

1. `abs_tol_umolkg` defaults to **5 µmol/kg** (vs Stage 3's 10) — this
   is a gate, not a screen.
2. `require_matching_units = True` — if the four species columns
   disagree on units OR fall outside `unit_equivalents`, the check is
   *not* attempted; the unit situation is flagged instead.

Three output flags:
- `flag_dic_species_audit_strict`        — species sum check failed
- `flag_dic_species_unit_mismatch_audit` — units disagree or off-list
- `flag_dic_species_unit_missing_audit`  — units missing


In [ ]:
dic_note = "Skipped (disabled by user or config)."
dic_result = pd.DataFrame()
dic_meta: dict = {}

dsa_cfg = config.get("dic_species_audit", {})
dic_check = DicSpeciesAudit(
    abs_tol_umolkg=float(dsa_cfg.get("abs_tol_umolkg", 5.0)),
    rel_tol=float(dsa_cfg.get("rel_tol", 0.01)),
    require_matching_units=bool(dsa_cfg.get("require_matching_units", True)),
)

if not NO_DIC_SPECIES_CHECK and bool(dsa_cfg.get("enabled", True)):
    dic_result, dic_note, dic_meta = dic_species_audit(
        df,
        dic_check,
        candidates=config.get("strict_dic_candidates", {}),
        unit_equivalents=set(config.get("unit_equivalents", [])),
    )
    for col in [
        "flag_dic_species_audit_strict",
        "flag_dic_species_unit_mismatch_audit",
        "flag_dic_species_unit_missing_audit",
    ]:
        df[col] = (
            dic_result[col] if col in dic_result.columns
            else pd.Series(False, index=df.index, dtype="boolean")
        )

    n_strict_fail = int(df["flag_dic_species_audit_strict"].fillna(False).sum())
    print(f"DIC audit strict failures: {n_strict_fail:,}")
else:
    notes.append("Strict DIC vs species audit disabled.")
    for col in [
        "flag_dic_species_audit_strict",
        "flag_dic_species_unit_mismatch_audit",
        "flag_dic_species_unit_missing_audit",
    ]:
        df[col] = pd.Series(False, index=df.index, dtype="boolean")

print(dic_note)

if display is not None and not dic_result.empty:
    display(dic_result.head(20))


## PASS / REVIEW / FAIL classification

The substantive new logic of Stage 4. `add_readiness_status` walks
every audit-side flag, assigns reason codes, and produces the
three-tier verdict. **FAIL** beats **REVIEW** beats **PASS** wherever
multiple tiers fire.

Three reason-code columns are written:

- `analysis_audit_reason_fail` — codes that triggered FAIL (or NA).
- `analysis_audit_reason_review` — codes that triggered REVIEW (or NA).
- `analysis_audit_reason_codes` — the union of the above; this is the
  column an analyst typically filters on.


In [ ]:
df = add_readiness_status(
    df, dup_table=dup_table, missing_key_idx=key_missing.index
)

status_counts = df["analysis_audit_status"].value_counts(dropna=False)
print(status_counts.to_string())

reason_counts_df = reason_count_table(df)
print("\nReason-code counts:")
print(reason_counts_df.to_string(index=False) if not reason_counts_df.empty else "  (none)")


## Quick preview

In [ ]:
preview_cols = [
    c for c in [
        "record_id", "sample_id", "sample_date", "sample_month",
        "station_id", "depth_round_m",
        "salinity", "temperature_insitu_c",
        "ta_best_umolkg", "ph_best", "ph_co2sys", "dic_best_umol_kg",
        "flag_any_carbonate_issue", "flag_any_carbonate_issue_strict",
        "range_flag_count", "flag_audit_strict_dic_fail",
        "analysis_audit_status", "analysis_audit_reason_codes",
    ]
    if c in df.columns
]
if display is not None:
    display(df[preview_cols].head(20))
else:
    print(df[preview_cols].head(20).to_string(index=False))


## Prepare output paths

```
<OUT_DIR>/
    data/
        analysis_ready.csv          # ◄── the final deliverable
        analysis_ready.parquet
    tables/
        column_inventory.csv
        missingness_top40.csv
        canonical_presence.csv
        alias_resolution.csv
        missing_key_rows.csv
        duplicates_by_keys.csv
        range_check_summary.csv     # (if range checks enabled)
        range_flags_long.csv        # (if range checks enabled, non-empty)
        dic_species_audit.csv       # (if DIC audit enabled, non-empty)
    reports/
        report.md
    logs/
        manifest.json
        effective_config.json
```


In [ ]:
paths = {
    "analysis_ready_csv":     dirs["data"]    / "analysis_ready.csv",
    "analysis_ready_parquet": dirs["data"]    / "analysis_ready.parquet",
    "column_inventory_csv":   dirs["tables"]  / "column_inventory.csv",
    "missingness_top40_csv":  dirs["tables"]  / "missingness_top40.csv",
    "canonical_presence_csv": dirs["tables"]  / "canonical_presence.csv",
    "alias_resolution_csv":   dirs["tables"]  / "alias_resolution.csv",
    "missing_key_rows_csv":   dirs["tables"]  / "missing_key_rows.csv",
    "duplicates_csv":         dirs["tables"]  / "duplicates_by_keys.csv",
    "range_summary_csv":      dirs["tables"]  / "range_check_summary.csv",
    "range_flags_csv":        dirs["tables"]  / "range_flags_long.csv",
    "dic_audit_csv":          dirs["tables"]  / "dic_species_audit.csv",
    "report_md":              dirs["reports"] / "report.md",
    "manifest_json":          dirs["logs"]    / "manifest.json",
    "effective_config_json":  dirs["logs"]    / "effective_config.json",
}
print(f"Output root: {out_root}")


## Write outputs

In [ ]:
parquet_written = False
parquet_error = None

if DRY_RUN:
    print("DRY_RUN = True -- no files written.")
else:
    inventory_df.to_csv(paths["column_inventory_csv"], index=False)
    inventory_df.head(40).to_csv(paths["missingness_top40_csv"], index=False)
    presence_df.to_csv(paths["canonical_presence_csv"], index=False)

    pd.DataFrame({
        "canonical_column": list(alias_resolution.keys()),
        "resolved_from": list(alias_resolution.values()),
    }).to_csv(paths["alias_resolution_csv"], index=False)

    key_missing.to_csv(paths["missing_key_rows_csv"], index=False)
    dup_table.to_csv(paths["duplicates_csv"], index=False)

    if not range_summary.empty:
        range_summary.to_csv(paths["range_summary_csv"], index=False)
    if not range_flags_df.empty:
        range_flags_df.to_csv(paths["range_flags_csv"], index=False)
    if not dic_result.empty:
        dic_result.to_csv(paths["dic_audit_csv"], index=False)

    if NO_PARQUET:
        df.to_csv(paths["analysis_ready_csv"], index=False)
        parquet_error = "Parquet disabled by user"
    else:
        parquet_written, parquet_error = write_csv_and_parquet(
            df, paths["analysis_ready_csv"], paths["analysis_ready_parquet"]
        )

    write_json(paths["effective_config_json"], config)

    # Build the report (inlined to avoid the three-same-name `write_report`
    # silent-overwrite hazard the original had).
    status_counts = (
        df["analysis_audit_status"].value_counts(dropna=False)
        .rename_axis("status").reset_index(name="count")
    )
    reason_counts_df = reason_count_table(df)
    notes_md = "\n".join(f"- {n}" for n in notes) if notes else "- (none)"
    notes_md = notes_md  # noqa: keep for f-string

    def _flag_count(col: str) -> int:
        if col in df.columns:
            return int(df[col].fillna(False).sum())
        return 0

    report_md_text = f"""# Stage 4 Analysis Audit Report

**Generated:** {utc_stamp()}
**Input:** `{input_csv}`
**Output root:** `{out_root}`
**Rows:** {len(df):,}  **Columns:** {df.shape[1]:,}

## What this stage does
Audits the Stage 3 table for contract compliance, key integrity, range
violations, and optional strict DIC species closure. Assigns
`analysis_audit_status` (PASS / REVIEW / FAIL) with explicit reason
codes. Does NOT rebuild chemistry or overwrite Stage 3 QC flags.

## Final readiness classification
{md_table_from_df(status_counts, max_rows=20)}

### Reason code counts
{md_table_from_df(reason_counts_df, max_rows=200) if not reason_counts_df.empty else "_(none)_"}

## Canonical field presence
{md_table_from_df(presence_df, max_rows=200)}

## Key integrity
- Duplicate keys requested: `{dup_keys}`
- Duplicate keys actually used: `{keys_used}`
- Rows with missing key fields: **{len(key_missing):,}**
- Duplicate rows found: **{len(dup_table):,}**

## Range checks
- Variables checked: **{len(range_summary):,}**
- Flagged cells: **{len(range_flags_df):,}**

{md_table_from_df(range_summary, max_rows=200) if not range_summary.empty else "_(skipped)_"}

## Strict DIC vs species audit
{dic_note}

- Rows with strict DIC fail: **{_flag_count("flag_audit_strict_dic_fail"):,}**
- Rows with DIC unit mismatch: **{_flag_count("flag_audit_dic_unit_mismatch"):,}**
- Rows with DIC unit missing: **{_flag_count("flag_audit_dic_unit_missing"):,}**

## Inherited Stage 3 context flags
| Flag | Rows |
|------|-----:|
| flag_any_carbonate_issue | {_flag_count("flag_any_carbonate_issue"):,} |
| flag_any_carbonate_issue_strict | {_flag_count("flag_any_carbonate_issue_strict"):,} |
| flag_stage2_replicate_conflict_carried | {_flag_count("flag_stage2_replicate_conflict_carried"):,} |
| flag_solver_unknown | {_flag_count("flag_solver_unknown"):,} |
| flag_carbon_input_pair_unknown | {_flag_count("flag_carbon_input_pair_unknown"):,} |

## Column inventory (top 30 by missingness)
{md_table_from_df(inventory_df.head(30), max_rows=200)}

## Notes
{notes_md}

## Main outputs
- analysis-ready CSV: `{paths["analysis_ready_csv"]}`
- range_flags_long: `{paths["range_flags_csv"]}`
- dic_species_audit: `{paths["dic_audit_csv"]}`
"""
    write_text(paths["report_md"], report_md_text)

    # Manifest
    manifest = {
        "notebook": "08_stage4",
        "generated_utc": utc_stamp(),
        "input_csv": str(input_csv),
        "output_root": str(out_root),
        "config_source": config_source,
        "parameters": {
            "INPUT_CSV": str(input_csv),
            "OUT_DIR": str(out_root),
            "CONFIG_PATH": CONFIG_PATH,
            "NO_RANGE_CHECKS": NO_RANGE_CHECKS,
            "NO_DIC_SPECIES_CHECK": NO_DIC_SPECIES_CHECK,
            "NO_PARQUET": NO_PARQUET,
            "DRY_RUN": DRY_RUN,
        },
        "policies": {
            "range_policy": asdict(policy),
            "dic_species_audit": asdict(dic_check),
            "required_stage3_cols": config.get("required_stage3_columns", []),
            "duplicate_keys": dup_keys,
            "duplicate_keys_used": keys_used,
        },
        "alias_resolution": alias_resolution,
        "dic_column_sources": dic_meta,
        "row_counts": {
            "n_rows": int(len(df)),
            "missing_key_rows": int(len(key_missing)),
            "duplicate_rows": int(len(dup_table)),
            "range_flagged_cells": int(len(range_flags_df)),
            "dic_strict_fails": _flag_count("flag_dic_species_audit_strict"),
            "status_PASS": int((df["analysis_audit_status"] == "PASS").sum()),
            "status_REVIEW": int((df["analysis_audit_status"] == "REVIEW").sum()),
            "status_FAIL": int((df["analysis_audit_status"] == "FAIL").sum()),
        },
        "reason_code_counts": (
            reason_counts_df.set_index("reason_code")["count"].to_dict()
            if not reason_counts_df.empty else {}
        ),
        "parquet_written": parquet_written,
        "parquet_error": parquet_error,
        "notes": notes,
        "outputs": {k: str(v) for k, v in paths.items()},
        "package_versions": {
            "python": sys.version.split()[0],
            "pandas": pd.__version__,
        },
    }
    write_json(paths["manifest_json"], manifest)

    print("\nStage 4 complete.")
    print(f"  -> analysis-ready CSV: {paths['analysis_ready_csv']}")


## Review written outputs

In [ ]:
if not DRY_RUN:
    outputs_df = pd.DataFrame(
        {"output_name": list(paths.keys()), "path": [str(p) for p in paths.values()]}
    )
    if display is not None:
        display(outputs_df)
        display(df[preview_cols].head(20))
    else:
        print(outputs_df.to_string(index=False))
